In [1]:
from collections import defaultdict
from scipy import stats
from scipy.stats import erlang, expon, norm 
from itertools import product
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import json

#### Extracting Global Time From Matrix?

# Tasks:

- Make infect time histogram
- Make path histogram
- Input data live Devlin's
- Distribution method like Devlin's 
- Generalize Edge Distribtion assignment (custom CDFS)
- Test Random Graphs, with n nodes, 1 is src and n is end, check if graph is connected
    - ER-model, parameterize n and p, 
    - if connected run 1000+ simulations
        - overall connected, we don't wnat to run simulations on small components
        - graph is connect if laplacian rank is n-1

In [12]:
class Graph(object):

    def __init__(self, edge_json, node_size=None, directed=False):
        # graph data structures
        self._graph = defaultdict(set)
        # Node information
        self._infected = defaultdict(lambda : False) # In our new rv algorithm, infected is a misnomer, it's just tracking if the rv has been "simulated" yet
        self._simulated = defaultdict(lambda : False) # In our new rv algorithm, infected is a misnomer, it's just tracking if the rv has been "simulated" yet
        self._parent = defaultdict(lambda : None)
        self._node_infect_time = defaultdict(lambda : 0) # to exploit "memoryless" exponential RV (s + t - s)
        self.edge_set = self.make_edge_set(edge_json)
        # Graph creation
        self._directed = directed
        self.add_connections(self.edge_set)
        self._adjency_matrix = self.construct_matrix(self.edge_set)
        
    def add_connections(self, edge_set):
        for node1, node2, wt in edge_set:
            self.add_edge(node1,node2, wt)
            
    def add_edge(self, src, dst, wt):
        self._graph[src].add((dst, wt))
        if self._directed == False:
            self._graph[dst].add((src, wt))
            
    def construct_matrix(self, edge_set):
        df = pd.DataFrame(edge_set)
        df = df.pivot(index=0, columns=1, values=2)
        if self._directed == False:
            df = df.combine_first(df.T)
        df = df.fillna(np.inf)
        return df

    # Make sure graph is using scipy.stats library
    def simulate_gossip_rv(self, src, dst):
        self.reset_simulation()

        src = np.array([src]).flatten() # transform scalars and lists to iterables
        dst = np.array([dst]).flatten()
        global_t = 0
        
        for node in src:
            self._infected[node] = True
            self._node_infect_time[node] = 0
            
        if ((g._adjency_matrix.loc[src] == np.inf).all()).all():
            raise ValueError(f"Source node {src} is not in the graph")
        
        terminate = False
        while not terminate:
            current_tick_infected = [key for key in self._infected.keys() if self._infected[key] == True]

            if ((g._adjency_matrix.loc[np.array(current_tick_infected)] == np.inf).all()).all():
                raise ValueError("No path to dst")

            min_edge = None
            min_infect_time = np.inf
            for infected in current_tick_infected:
                # simulate new frontier infections
                for col, rv in enumerate(self._adjency_matrix.loc[infected]):
                    new_infection = self._adjency_matrix.columns[col]
                    path = self._adjency_matrix.loc[infected, new_infection]
                    # check if node has not been simulated/infected
                    if not self._simulated[(infected, new_infection)] and path != np.inf:
                        self._simulated[(infected, new_infection)] = True
                        edge_delay = rv.rvs() # scipy.stats dependency
                        self._adjency_matrix.loc[infected, new_infection] = edge_delay
                        if (self._directed == False):
                            self._adjency_matrix.loc[new_infection, infected] = edge_delay
                            self._simulated[(new_infection, infected)] = True
                        display(self._adjency_matrix)
                    if self._adjency_matrix.loc[infected, new_infection] < min_infect_time:
                        min_infect_time = self._adjency_matrix.loc[infected, new_infection] 
                        min_edge = (infected, new_infection)
            if self._parent[min_edge[1]] == None:
                self._parent[min_edge[1]] = min_edge[0]
                self._node_infect_time[min_edge[1]] = min_infect_time
                self._infected[min_edge[1]] = True
            self._adjency_matrix.loc[min_edge[0], min_edge[1]] = np.inf
            self._adjency_matrix.loc[current_tick_infected] = self._adjency_matrix.loc[current_tick_infected].sub(min_infect_time)
            if self._directed == False:
                self._adjency_matrix.loc[min_edge[1], min_edge[0]] = np.inf
                self._adjency_matrix.loc[:, current_tick_infected] = self._adjency_matrix.loc[:, current_tick_infected].sub(min_infect_time)
            display(self._adjency_matrix)
            global_t = global_t + min_infect_time
            for node in dst:
                if self._infected[node]:
                    return global_t

    def reset_simulation(self):
        keys = self._graph.keys()
        for key in keys:
            self._infected[key] = False
            self._parent[key] = None
            self._node_infect_time[key] = 0
        for edge in product(keys, keys):
            self._simulated[edge] = False
        self._adjency_matrix = self.construct_matrix(self.edge_set)

    def construct_path(self, dst):
        path = []
        curr_node = dst
        while curr_node is not None:
            path.append(curr_node)
            curr_node = self._parent[curr_node]
        
        return path
    
    def make_edge_set(self, edge_json):
        edge_set = set()
        for key, value in edge_json.items():
            edges = key.split(',')
            distribution = self.process_distribution_params(value)
            edge_tuple = (edges[0], edges[1], distribution)
            edge_set.add(edge_tuple)
        return edge_set
                               
    def process_distribution_params(self, function_dict):
        distribution_map = {
            "E" : expon, # assuming params lambda = 1.0
            "N" : norm, # assuming unit normal
            "custom" : None, # customRV, # not working
        }
        distribution = distribution_map[function_dict["distribution"]]
        return distribution
                               


In [13]:
exp_t = lambda t: 1 - np.e**(-t)

edge_set = {
    "1,2": { "distribution": "E", "parameters": { "lambda" : 1.0 }}, 
    "1,3": { "distribution": "E", "parameters": { "lambda" : 1.0 }}, 
    "2,4": { "distribution": "E", "parameters": { "lambda" : 1.0 }}, 
    "2,5": { "distribution": "E", "parameters": { "lambda" : 1.0 }}, 
    "3,6": { "distribution": "E", "parameters": { "lambda" : 1.0 }}, 
    "4,5": { "distribution": "E", "parameters": { "lambda" : 1.0 }}, 
    "4,6": { "distribution": "E", "parameters": { "lambda" : 1.0 }}, 
}

g = Graph(edge_set)

In [14]:
((g._adjency_matrix.loc[np.array(['1','2'])] == np.inf).all()).all()

np.False_

#### Note: Columns don't actually do a good job of differentiating if a column is a string or int

In [15]:
g.reset_simulation()
g._adjency_matrix

,1,2,3,4,5,6
1,inf,<scipy.stats._continuous_distns.expon_gen obje...,<scipy.stats._continuous_distns.expon_gen obje...,inf,inf,inf
2,<scipy.stats._continuous_distns.expon_gen obje...,inf,inf,<scipy.stats._continuous_distns.expon_gen obje...,<scipy.stats._continuous_distns.expon_gen obje...,inf
3,<scipy.stats._continuous_distns.expon_gen obje...,inf,inf,inf,inf,<scipy.stats._continuous_distns.expon_gen obje...
4,inf,<scipy.stats._continuous_distns.expon_gen obje...,inf,inf,<scipy.stats._continuous_distns.expon_gen obje...,<scipy.stats._continuous_distns.expon_gen obje...
5,inf,<scipy.stats._continuous_distns.expon_gen obje...,inf,<scipy.stats._continuous_distns.expon_gen obje...,inf,inf
6,inf,inf,<scipy.stats._continuous_distns.expon_gen obje...,<scipy.stats._continuous_distns.expon_gen obje...,inf,inf


In [17]:
t = g.simulate_gossip_rv('1','6')
path = g.construct_path('6')
display(path)
print(f"Time: {t}")

,1,2,3,4,5,6
1,inf,1.228101,<scipy.stats._continuous_distns.expon_gen obje...,inf,inf,inf
2,1.228101,inf,inf,<scipy.stats._continuous_distns.expon_gen obje...,<scipy.stats._continuous_distns.expon_gen obje...,inf
3,<scipy.stats._continuous_distns.expon_gen obje...,inf,inf,inf,inf,<scipy.stats._continuous_distns.expon_gen obje...
4,inf,<scipy.stats._continuous_distns.expon_gen obje...,inf,inf,<scipy.stats._continuous_distns.expon_gen obje...,<scipy.stats._continuous_distns.expon_gen obje...
5,inf,<scipy.stats._continuous_distns.expon_gen obje...,inf,<scipy.stats._continuous_distns.expon_gen obje...,inf,inf
6,inf,inf,<scipy.stats._continuous_distns.expon_gen obje...,<scipy.stats._continuous_distns.expon_gen obje...,inf,inf


,1,2,3,4,5,6
1,inf,1.228101,0.997929,inf,inf,inf
2,1.228101,inf,inf,<scipy.stats._continuous_distns.expon_gen obje...,<scipy.stats._continuous_distns.expon_gen obje...,inf
3,0.997929,inf,inf,inf,inf,<scipy.stats._continuous_distns.expon_gen obje...
4,inf,<scipy.stats._continuous_distns.expon_gen obje...,inf,inf,<scipy.stats._continuous_distns.expon_gen obje...,<scipy.stats._continuous_distns.expon_gen obje...
5,inf,<scipy.stats._continuous_distns.expon_gen obje...,inf,<scipy.stats._continuous_distns.expon_gen obje...,inf,inf
6,inf,inf,<scipy.stats._continuous_distns.expon_gen obje...,<scipy.stats._continuous_distns.expon_gen obje...,inf,inf


,1,2,3,4,5,6
1,inf,0.230171,inf,inf,inf,inf
2,0.230171,inf,inf,<scipy.stats._continuous_distns.expon_gen obje...,<scipy.stats._continuous_distns.expon_gen obje...,inf
3,inf,inf,inf,inf,inf,<scipy.stats._continuous_distns.expon_gen obje...
4,inf,<scipy.stats._continuous_distns.expon_gen obje...,inf,inf,<scipy.stats._continuous_distns.expon_gen obje...,<scipy.stats._continuous_distns.expon_gen obje...
5,inf,<scipy.stats._continuous_distns.expon_gen obje...,inf,<scipy.stats._continuous_distns.expon_gen obje...,inf,inf
6,inf,inf,<scipy.stats._continuous_distns.expon_gen obje...,<scipy.stats._continuous_distns.expon_gen obje...,inf,inf


,1,2,3,4,5,6
1,inf,0.230171,inf,inf,inf,inf
2,0.230171,inf,inf,<scipy.stats._continuous_distns.expon_gen obje...,<scipy.stats._continuous_distns.expon_gen obje...,inf
3,inf,inf,inf,inf,inf,1.297968
4,inf,<scipy.stats._continuous_distns.expon_gen obje...,inf,inf,<scipy.stats._continuous_distns.expon_gen obje...,<scipy.stats._continuous_distns.expon_gen obje...
5,inf,<scipy.stats._continuous_distns.expon_gen obje...,inf,<scipy.stats._continuous_distns.expon_gen obje...,inf,inf
6,inf,inf,1.297968,<scipy.stats._continuous_distns.expon_gen obje...,inf,inf


,1,2,3,4,5,6
1,inf,inf,inf,inf,inf,inf
2,inf,inf,inf,<scipy.stats._continuous_distns.expon_gen obje...,<scipy.stats._continuous_distns.expon_gen obje...,inf
3,inf,inf,inf,inf,inf,1.067797
4,inf,<scipy.stats._continuous_distns.expon_gen obje...,inf,inf,<scipy.stats._continuous_distns.expon_gen obje...,<scipy.stats._continuous_distns.expon_gen obje...
5,inf,<scipy.stats._continuous_distns.expon_gen obje...,inf,<scipy.stats._continuous_distns.expon_gen obje...,inf,inf
6,inf,inf,1.067797,<scipy.stats._continuous_distns.expon_gen obje...,inf,inf


...

,1,2,3,4,5,6
1,inf,inf,inf,inf,inf,inf
2,inf,inf,inf,1.638895,inf,inf
3,inf,inf,inf,inf,inf,0.737415
4,inf,1.638895,inf,inf,<scipy.stats._continuous_distns.expon_gen obje...,<scipy.stats._continuous_distns.expon_gen obje...
5,inf,inf,inf,<scipy.stats._continuous_distns.expon_gen obje...,inf,inf
6,inf,inf,0.737415,<scipy.stats._continuous_distns.expon_gen obje...,inf,inf


,1,2,3,4,5,6
1,inf,inf,inf,inf,inf,inf
2,inf,inf,inf,1.638895,inf,inf
3,inf,inf,inf,inf,inf,0.737415
4,inf,1.638895,inf,inf,0.793832,<scipy.stats._continuous_distns.expon_gen obje...
5,inf,inf,inf,0.793832,inf,inf
6,inf,inf,0.737415,<scipy.stats._continuous_distns.expon_gen obje...,inf,inf


,1,2,3,4,5,6
1,inf,inf,inf,inf,inf,inf
2,inf,inf,inf,0.90148,inf,inf
3,inf,inf,inf,inf,inf,inf
4,inf,0.90148,inf,inf,0.056418,<scipy.stats._continuous_distns.expon_gen obje...
5,inf,inf,inf,0.056418,inf,inf
6,inf,inf,inf,<scipy.stats._continuous_distns.expon_gen obje...,inf,inf


['6', '3', '1']

Time: 2.2958973115182353


In [7]:
g._adjency_matrix

,1,2,3,4,5,6
1,inf,inf,inf,inf,inf,inf
2,inf,inf,inf,0.399514,0.179455,inf
3,inf,inf,inf,inf,inf,inf
4,inf,0.399514,inf,inf,<scipy.stats._continuous_distns.expon_gen obje...,<scipy.stats._continuous_distns.expon_gen obje...
5,inf,0.179455,inf,<scipy.stats._continuous_distns.expon_gen obje...,inf,inf
6,inf,inf,inf,<scipy.stats._continuous_distns.expon_gen obje...,inf,inf
